# h2ml Quickstart

This notebook demonstrates the h2ml pipeline on two sklearn toy datasets:

1. **Classification** — breast cancer dataset (binary, 30 features)
2. **Regression** — diabetes dataset (10 features, y-transform sweep)

No external data required — everything runs with `sklearn.datasets`.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer, load_diabetes

from h2ml.features.feature_store import PipelineData
from h2ml.pipeline.pipeline import H2MLPipeline, PipelineConfig
from h2ml.plots import (
    cv_diagnostics,
    pipeline_scores,
    shap_dependence,
    shap_importance,
    shap_summary_plot,
)

---
## 1. Classification — breast cancer dataset

In [ ]:
data = load_breast_cancer()

store_clf = PipelineData(
    X=data.data.astype(np.float32),
    feature_names=list(data.feature_names),
    y=data.target.astype(np.float32),
)

print(store_clf)

In [ ]:
config_clf = PipelineConfig(
    task_type="classification",   # or "regression"; the TaskType enum is also accepted
    metric="AUC",
    n_splits=5,
    n_trials=30,
    verbose=True,
)

pipeline_clf = H2MLPipeline(config=config_clf)
result_clf = pipeline_clf.run(store_clf)

In [ ]:
# Results across all pipeline stages
result_clf.summary(metric="AUC_Test_Mean")

In [ ]:
# Features selected after step 2
print(f"Original features : {store_clf.n_features}")
print(f"Reduced features  : {result_clf.features_reduced.n_features}")
print(f"Kept              : {result_clf.features_reduced.feature_names}")

In [ ]:
# SHAP feature importance from step 2
result_clf.selector.importance_summary()

In [ ]:
# Build and inspect the deployment artifact
final_clf = result_clf.build_final_model()
print(final_clf)

# Predict on the training set as a sanity check
preds = final_clf.predict(store_clf.X[:, [store_clf.feature_names.index(f) for f in final_clf.feature_names]])
print(f"Prediction sample : {preds[:10]}")

### Conformal prediction sets (classification)

`build_final_model()` automatically calibrates a conformal predictor from the out-of-fold CV predictions. `predict_set(X, alpha)` returns, for each sample, the set of classes that are plausible at the requested coverage level.

- **`[1]`** or **`[0]`** — model is confident; only one class is compatible with the coverage guarantee  
- **`[0, 1]`** — model is uncertain; both classes are plausible  

Coverage is guaranteed to be ≥ `1 - alpha` on average over new draws from the training distribution.

In [ ]:
# Align X to the features the final model was trained on
feat_idx = [store_clf.feature_names.index(f) for f in final_clf.feature_names]
X_aligned = store_clf.X[:, feat_idx]

# 90% conformal prediction sets
sets = final_clf.predict_set(X_aligned, alpha=0.10)

n_confident = sum(len(s) == 1 for s in sets)
n_uncertain = sum(len(s) == 2 for s in sets)
n_empty = sum(len(s) == 0 for s in sets)

print(f"Confident   (singleton set): {n_confident} / {len(sets)}")
print(f"Uncertain   (both classes) : {n_uncertain} / {len(sets)}")
print(f"Empty set                  : {n_empty} / {len(sets)}")
print(f"\nFirst 10 prediction sets: {[s.tolist() for s in sets[:10]]}")
print(f"Calibration threshold q  : {final_clf.conformal.threshold(0.10):.4f}")

### Diagnostic plots

In [ ]:
pipeline_scores(result_clf)
cv_diagnostics(result_clf.best_cv_result)
shap_importance(result_clf.selector)
shap_summary_plot(result_clf)
shap_dependence(result_clf, n_features=6)

---
## 2. Regression — diabetes dataset with y-transform sweep

In [ ]:
data_r = load_diabetes()

store_reg = PipelineData(
    X=data_r.data.astype(np.float32),
    feature_names=list(data_r.feature_names),
    y=data_r.target.astype(np.float32),
)

print(store_reg)

In [ ]:
config_reg = PipelineConfig(
    task_type="regression",
    metric="R2",
    n_splits=5,
    n_trials=30,
    verbose=True,
)

pipeline_reg = H2MLPipeline(config=config_reg)

# Sweep log, sqrt, and identity transforms alongside raw y
result_reg = pipeline_reg.run(store_reg, transforms=["count", "log", "sqrt"])

### Conformal prediction intervals (regression)

`predict_interval(X, alpha)` returns symmetric `(lower, upper)` bounds around the point estimate. The interval width is constant (`2q`) — the same threshold `q` applies to every prediction.

**Note:** if a y-transform was used, the interval is in the transformed space. Apply the inverse transform to the bounds to recover original-scale intervals.

In [ ]:
final_reg = result_reg.build_final_model()

feat_idx_r = [store_reg.feature_names.index(f) for f in final_reg.feature_names]
X_aligned_r = store_reg.X[:, feat_idx_r]

# 90% prediction intervals
lower, upper = final_reg.predict_interval(X_aligned_r, alpha=0.10)
y_hat = final_reg.predict(X_aligned_r)

print(f"Calibration threshold q : {final_reg.conformal.threshold(0.10):.4f}")
print(f"Interval half-width     : ±{(upper - lower).mean() / 2:.4f}")
print()

# If a y-transform was applied, optionally invert the bounds
if result_reg.y_transform:
    from h2ml.preprocessing.transforms import INVERSE_TRANSFORMS

    inv = INVERSE_TRANSFORMS.get(result_reg.y_transform)
    if inv is not None:
        lower_orig = inv(lower)
        upper_orig = inv(upper)
        y_hat_orig = inv(y_hat)
        print(f"Transform in use        : {result_reg.y_transform}")
        print(f"Original-scale interval : [{lower_orig[:3].round(1)} … {upper_orig[:3].round(1)}]")
    else:
        print(f"No inverse transform registered for '{result_reg.y_transform}'")
else:
    print("Sample intervals (first 5 rows):")
    for i in range(5):
        print(f"  ŷ={y_hat[i]:.2f}  [{lower[i]:.2f}, {upper[i]:.2f}]")

In [ ]:
result_reg.summary(metric="R2_Test_Mean")

In [ ]:
print(f"Best model     : {result_reg.best_model_name}")
print(f"Best stage     : {result_reg.best_stage}")
print(f"Best transform : {result_reg.y_transform}")
print(f"Best params    : {result_reg.best_params}")

### Diagnostic plots

In [ ]:
pipeline_scores(result_reg)
cv_diagnostics(result_reg.best_cv_result)
shap_importance(result_reg.selector)
shap_summary_plot(result_reg)
shap_dependence(result_reg, n_features=6)

---
## 3. Partial run — step 1 only (quick model screening)

In [ ]:
# Useful when you just want to compare models before committing to the full pipeline
result_screen = pipeline_clf.run_step1_only(store_clf)
result_screen.step1_agg_df.sort_values("AUC_Test_Mean", ascending=False)

---
## 4. Persistence

In [ ]:
from pathlib import Path

out = Path("outputs")
out.mkdir(exist_ok=True)

# Save the full pipeline result
result_clf.save(out / "clf_result")

# Save the deployment model
final_clf.save(out / "clf_final_model.pkl")

# Reload
from h2ml.pipeline.pipeline import PipelineResult  # noqa: E402
from h2ml.pipeline.final_model import FinalModel  # noqa: E402

result_reloaded = PipelineResult.load(out / "clf_result")
model_reloaded = FinalModel.load(out / "clf_final_model.pkl")

print(result_reloaded)
print(model_reloaded)